# Phase 3.II — Benchmark bộ test mentor/sếp trên máy local

Notebook này chỉ giữ lại phần **Phase 3.II — Mentor-provided test benchmark**.
Mục tiêu là để anh chạy trực tiếp trên máy local với bộ test có cấu trúc:

```text
testset/
├── transcript.txt
└── audio/
    ├── *.wav
    └── ...
```

Notebook này **không train**, **không xử lý ViMedCSS nội bộ**, **không phụ thuộc Google Drive/Colab**.

Luồng xử lý chính:

```text
kiểm tra cấu trúc testset
→ chuẩn hóa transcript/audio thành manifest
→ chạy ASR inference bằng model được chỉ định
→ xuất prediction.tsv
→ chạy benchmark WER/CER/CS-WER/negation miss rate
→ tạo report Markdown để lưu kết quả
```

Anh chỉ cần chỉnh 2 biến quan trọng ở cell cấu hình:

```python
TESTSET_ROOT = Path(r"...")
MODEL_PATH = r"..."
```


## 0. Ghi chú vận hành

- Notebook ưu tiên **log rõ ràng trong lúc chạy**: mỗi vài file sẽ in tiến độ, thời gian xử lý và preview prediction.
- Nếu inference bị dừng giữa chừng, chạy lại cell inference với `RESUME = True` để tiếp tục từ file chưa xử lý.
- Output mặc định được lưu trong `mentor_test_benchmark_outputs/`.
- Benchmark dùng format `filename<TAB>text`.


In [ ]:
# ============================================================
# 1. CẤU HÌNH CHÍNH — Anh chỉ cần sửa block này
# ============================================================
from pathlib import Path

# Thư mục testset trên máy của anh.
# Ví dụ Linux:   Path(r"/home/user/data/testset")
# Ví dụ Windows: Path(r"D:\\data\\testset")
TESTSET_ROOT = Path(r"/path/to/testset")

# Model dùng để benchmark.
# Có thể là đường dẫn local tới checkpoint hoặc Hugging Face model id.
MODEL_PATH = r"/path/to/phowhisper_vietmed_week5_checkpoint"

RUN_LABEL = "week5_vietmed_checkpoint_on_mentor_test"
OUTPUT_ROOT = None

SMOKE_MAX_SAMPLES = 3
MAX_SAMPLES_FOR_FULL_RUN = None  # None = chạy toàn bộ; số nguyên = chỉ chạy N mẫu đầu.
RESUME = True
LOG_EVERY = 5

ASR_LANGUAGE = "vi"
ASR_TASK = "transcribe"

# Optional: nếu có file syllables, benchmark sẽ tính thêm CS-WER/N-WER.
VIET_SYLLABLES_PATH = None

print("TESTSET_ROOT =", TESTSET_ROOT)
print("MODEL_PATH   =", MODEL_PATH)
print("RUN_LABEL    =", RUN_LABEL)


In [ ]:
# ============================================================
# 2. KIỂM TRA DEPENDENCIES VÀ THIẾT BỊ
# ============================================================
import sys
import platform
import importlib.util

required_packages = ["torch", "transformers", "soundfile", "tqdm"]
missing = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]

print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())
print("Missing required packages:", missing if missing else "None")

if missing:
    print("\nCần cài dependencies trước khi chạy tiếp:")
    print("%pip install -U torch transformers accelerate soundfile tqdm pandas")
    raise RuntimeError("Thiếu package bắt buộc. Cài package rồi chạy lại cell này.")

import torch
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("[WARN] Không thấy CUDA. Notebook vẫn chạy được trên CPU nhưng ASR inference sẽ chậm.")


### Optional — Cài dependencies

Chỉ chạy cell dưới nếu cell kiểm tra dependencies báo thiếu package.


In [ ]:
# %pip install -U torch transformers accelerate soundfile tqdm pandas


In [ ]:
# ============================================================
# 3. TẠO OUTPUT DIRECTORY VÀ KIỂM TRA CẤU TRÚC TESTSET
# ============================================================
from pathlib import Path
import json
import os
from datetime import datetime, timezone

TESTSET_ROOT = Path(TESTSET_ROOT).expanduser().resolve()
AUDIO_DIR = TESTSET_ROOT / "audio"
TRANSCRIPT_PATH = TESTSET_ROOT / "transcript.txt"

if OUTPUT_ROOT is None:
    OUTPUT_ROOT = Path.cwd() / "mentor_test_benchmark_outputs"
else:
    OUTPUT_ROOT = Path(OUTPUT_ROOT).expanduser().resolve()

RUN_DIR = OUTPUT_ROOT / RUN_LABEL
REF_DIR = RUN_DIR / "reference"
PRED_DIR = RUN_DIR / "predictions"
REPORT_DIR = RUN_DIR / "reports"
LOG_DIR = RUN_DIR / "logs"

for d in [OUTPUT_ROOT, RUN_DIR, REF_DIR, PRED_DIR, REPORT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("=== PATH CONFIG ===")
print("TESTSET_ROOT    =", TESTSET_ROOT)
print("AUDIO_DIR       =", AUDIO_DIR)
print("TRANSCRIPT_PATH =", TRANSCRIPT_PATH)
print("RUN_DIR         =", RUN_DIR)
print("MODEL_PATH      =", MODEL_PATH)

errors = []
if not TESTSET_ROOT.exists(): errors.append(f"TESTSET_ROOT không tồn tại: {TESTSET_ROOT}")
if not AUDIO_DIR.exists(): errors.append(f"Không tìm thấy thư mục audio/: {AUDIO_DIR}")
if not TRANSCRIPT_PATH.exists(): errors.append(f"Không tìm thấy transcript.txt: {TRANSCRIPT_PATH}")
if errors:
    print("\n[FAIL] Lỗi cấu trúc input:")
    for e in errors: print("-", e)
    raise FileNotFoundError("Cấu trúc testset chưa đúng. Sửa TESTSET_ROOT rồi chạy lại.")

wav_files = sorted(AUDIO_DIR.glob("*.wav"))
print("\n=== INPUT SUMMARY ===")
print("Số file .wav:", len(wav_files))
print("transcript.txt size:", TRANSCRIPT_PATH.stat().st_size, "bytes")
if not wav_files:
    raise RuntimeError(f"Không có file .wav trong {AUDIO_DIR}")

print("\nPreview audio files:")
for p in wav_files[:10]: print("-", p.name)

print("\nPreview transcript.txt:")
with TRANSCRIPT_PATH.open("r", encoding="utf-8-sig", errors="replace") as f:
    for i, line in zip(range(10), f):
        print(line.rstrip("\n")[:300])


In [ ]:
# ============================================================
# 4. CHUẨN HÓA transcript.txt + audio/*.wav THÀNH MANIFEST/REFERENCE TSV
# ============================================================
import re
import csv
import json
from pathlib import Path
from typing import Any


def norm_space(s: Any) -> str:
    return re.sub(r"\s+", " ", str(s).strip())


def clean_tsv_cell(s: Any) -> str:
    return norm_space(str(s if s is not None else "").replace("\t", " ").replace("\n", " "))


def read_transcript_txt(path: Path) -> dict[str, str]:
    # Ưu tiên format: filename<TAB>text. Fallback: filename text...
    refs: dict[str, str] = {}
    with path.open("r", encoding="utf-8-sig", errors="replace") as f:
        lines = [line.rstrip("\n") for line in f if line.strip()]
    tab_lines = [line for line in lines if "\t" in line]
    if tab_lines:
        for line in tab_lines:
            key, text = line.split("\t", 1)
            key = norm_space(key); text = clean_tsv_cell(text)
            if key and text: refs[key] = text
        return refs
    for line in lines:
        parts = line.split(maxsplit=1)
        if len(parts) == 2:
            key, text = parts
            key = norm_space(key); text = clean_tsv_cell(text)
            if key and text: refs[key] = text
    return refs


def key_variants(key: str) -> list[str]:
    raw = str(key).strip().replace("\\", "/")
    p = Path(raw)
    name, stem = p.name, p.stem
    variants = {raw, raw.lower(), name, name.lower(), stem, stem.lower(), f"{stem}.wav", f"{stem.lower()}.wav"}
    return [v for v in variants if v]


def build_audio_index(audio_files: list[Path]) -> dict[str, Path]:
    index: dict[str, Path] = {}
    for p in audio_files:
        for k in {p.name, p.name.lower(), p.stem, p.stem.lower()}:
            index.setdefault(k, p)
    return index


def find_audio_for_key(key: str, audio_index: dict[str, Path]) -> Path | None:
    for v in key_variants(key):
        if v in audio_index: return audio_index[v]
    return None


audio_files = sorted(AUDIO_DIR.glob("*.wav"))
refs = read_transcript_txt(TRANSCRIPT_PATH)
audio_index = build_audio_index(audio_files)
matched_rows, unmatched_refs = [], []
used_audio = set()

for ref_key, text in refs.items():
    audio_path = find_audio_for_key(ref_key, audio_index)
    if audio_path is None:
        unmatched_refs.append({"ref_key": ref_key, "text": text})
        continue
    sample_id = audio_path.name
    used_audio.add(str(audio_path.resolve()))
    matched_rows.append({
        "sample_id": sample_id,
        "audio": str(audio_path.resolve()),
        "reference_text": text,
        "original_ref_key": ref_key,
        "audio_basename": audio_path.name,
        "audio_stem": audio_path.stem,
    })

unreferenced_audio = [p for p in audio_files if str(p.resolve()) not in used_audio]

manifest_path = REF_DIR / "mentor_test_manifest.jsonl"
reference_tsv_path = REF_DIR / "mentor_test_reference.tsv"
inventory_path = REF_DIR / "mentor_test_inventory.csv"
summary_path = REF_DIR / "mentor_test_prepare_summary.json"
prepare_report_path = REPORT_DIR / "MENTOR_TEST_PREPARE_REPORT.md"

with manifest_path.open("w", encoding="utf-8") as f:
    for row in matched_rows: f.write(json.dumps(row, ensure_ascii=False) + "\n")
with reference_tsv_path.open("w", encoding="utf-8") as f:
    for row in matched_rows: f.write(f"{clean_tsv_cell(row['sample_id'])}\t{clean_tsv_cell(row['reference_text'])}\n")
with inventory_path.open("w", encoding="utf-8", newline="") as f:
    w = csv.writer(f); w.writerow(["kind", "path", "size_bytes"])
    for p in audio_files: w.writerow(["audio", str(p), p.stat().st_size])
    w.writerow(["transcript", str(TRANSCRIPT_PATH), TRANSCRIPT_PATH.stat().st_size])

summary = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "testset_root": str(TESTSET_ROOT),
    "audio_dir": str(AUDIO_DIR),
    "transcript_path": str(TRANSCRIPT_PATH),
    "n_audio_files": len(audio_files),
    "n_transcript_rows": len(refs),
    "n_matched": len(matched_rows),
    "n_unmatched_refs": len(unmatched_refs),
    "n_unreferenced_audio": len(unreferenced_audio),
    "manifest_jsonl": str(manifest_path),
    "reference_tsv": str(reference_tsv_path),
    "inventory_csv": str(inventory_path),
    "unmatched_refs_preview": unmatched_refs[:20],
    "unreferenced_audio_preview": [p.name for p in unreferenced_audio[:20]],
}
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

report = f"""# Mentor Test Set Preparation Report

## Mục tiêu
Chuẩn hóa bộ test mentor/sếp cung cấp thành manifest và reference TSV để chạy ASR benchmark.

## Input
- Testset root: `{TESTSET_ROOT}`
- Audio dir: `{AUDIO_DIR}`
- Transcript: `{TRANSCRIPT_PATH}`

## Kết quả khớp dữ liệu
| Hạng mục | Số lượng |
|---|---:|
| Audio `.wav` tìm thấy | {len(audio_files)} |
| Dòng transcript đọc được | {len(refs)} |
| Audio/transcript khớp được | {len(matched_rows)} |
| Transcript không khớp audio | {len(unmatched_refs)} |
| Audio không có transcript | {len(unreferenced_audio)} |

## Output
- Manifest JSONL: `{manifest_path}`
- Reference TSV: `{reference_tsv_path}`
- Inventory CSV: `{inventory_path}`
- Summary JSON: `{summary_path}`

## Quyết định
{'PASS: Có thể chạy inference.' if matched_rows else 'FAIL: Không có cặp audio/transcript nào khớp. Cần kiểm tra transcript.txt.'}
"""
prepare_report_path.write_text(report, encoding="utf-8")

print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\n=== Reference TSV preview ===")
print(reference_tsv_path.read_text(encoding="utf-8").splitlines()[:5])
if not matched_rows:
    raise RuntimeError("Không có cặp audio/transcript khớp. Kiểm tra transcript.txt và tên file .wav.")


In [ ]:
# ============================================================
# 5. KIỂM TRA NHANH DURATION CỦA AUDIO VÀ PREVIEW MANIFEST
# ============================================================
import soundfile as sf
import json
from pathlib import Path

rows = [json.loads(line) for line in manifest_path.read_text(encoding="utf-8").splitlines() if line.strip()]
print("Số mẫu trong manifest:", len(rows))
print("\nManifest preview:")
for row in rows[:3]: print(json.dumps(row, ensure_ascii=False, indent=2)[:1000])

print("\nAudio duration preview:")
durations = []
for row in rows[: min(20, len(rows))]:
    audio = Path(row["audio"])
    try:
        info = sf.info(str(audio))
        dur = info.frames / float(info.samplerate)
        durations.append(dur)
        print(f"{audio.name:40s} | {dur:8.2f}s | sr={info.samplerate} | ch={info.channels}")
    except Exception as e:
        print(f"[WARN] Cannot read {audio}: {e}")
if durations:
    print("\nDuration preview stats:", "min", round(min(durations), 3), "max", round(max(durations), 3), "mean", round(sum(durations)/len(durations), 3))


In [ ]:
# ============================================================
# 6. BENCHMARK FUNCTIONS: WER/CER/CS-WER/NEGATION MISS RATE
# ============================================================
import re
import json
import unicodedata
from collections import Counter
from pathlib import Path
from typing import Any

NEGATION_WORDS = {"không", "chưa", "chẳng", "chả", "đừng", "chớ"}
EN_TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z0-9_\-+/]*")
PUNCT_RE = re.compile(r"[^\w\sÀ-ỹà-ỹđĐ]", flags=re.UNICODE)


def normalize_for_metric(text: str, remove_punct: bool = False) -> str:
    text = unicodedata.normalize("NFC", str(text or "")).lower().replace("\t", " ").replace("\n", " ")
    if remove_punct: text = PUNCT_RE.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()


def levenshtein(ref: list[str], hyp: list[str]) -> tuple[int, int, int]:
    n, m = len(ref), len(hyp)
    dp = [[(0, 0, 0, 0)] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1): dp[i][0] = (i, 0, i, 0)
    for j in range(1, m + 1): dp[0][j] = (j, 0, 0, j)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref[i - 1] == hyp[j - 1]: dp[i][j] = dp[i - 1][j - 1]
            else:
                sub_cost, sub_s, sub_d, sub_i = dp[i - 1][j - 1]
                del_cost, del_s, del_d, del_i = dp[i - 1][j]
                ins_cost, ins_s, ins_d, ins_i = dp[i][j - 1]
                dp[i][j] = min((sub_cost+1, sub_s+1, sub_d, sub_i), (del_cost+1, del_s, del_d+1, del_i), (ins_cost+1, ins_s, ins_d, ins_i+1))
    _, s, d, ins = dp[n][m]
    return s, d, ins


def load_tsv(path: Path) -> dict[str, str]:
    data = {}
    with path.open("r", encoding="utf-8", errors="replace") as f:
        for lineno, line in enumerate(f, start=1):
            line = line.rstrip("\n")
            if not line.strip(): continue
            parts = line.split("\t", 1)
            if len(parts) < 2:
                print(f"[WARN] {path}:{lineno} không đủ 2 cột, bỏ qua")
                continue
            data[parts[0].strip()] = parts[1].strip()
    return data


def load_viet_syllables(path: str | Path | None) -> frozenset[str] | None:
    if not path: return None
    p = Path(path).expanduser()
    if not p.exists():
        print(f"[WARN] Không tìm thấy viet syllables file: {p}")
        return None
    values = []
    with p.open("r", encoding="utf-8", errors="replace") as f:
        for line in f:
            v = normalize_for_metric(line.strip(), remove_punct=True)
            if v: values.append(v)
    print(f"[OK] Loaded Vietnamese syllables: {len(values)} entries from {p}")
    return frozenset(values)


def is_non_vietnamese_word(word: str, viet_set: frozenset[str]) -> bool:
    w = normalize_for_metric(word, remove_punct=True)
    return bool(w and w not in viet_set)


def english_tokens(words: list[str]) -> list[str]:
    return [w for w in words if EN_TOKEN_RE.search(w)]


def compute_metrics(references: dict[str, str], predictions: dict[str, str], viet_set: frozenset[str] | None = None, remove_punct: bool = False) -> dict[str, Any]:
    common_keys = sorted(set(references) & set(predictions))
    missing_in_pred = sorted(set(references) - set(predictions))
    missing_in_ref = sorted(set(predictions) - set(references))
    total_s = total_d = total_i = total_ref_words = 0
    char_s = char_d = char_i = total_ref_chars = 0
    en_s = en_d = en_i = en_ref_words = 0
    cs_s = cs_d = cs_i = cs_ref_words = 0
    n_s = n_d = n_i = n_ref_words = 0
    neg_missed = neg_ref_count = neg_utterances = 0
    per_sample = []
    for key in common_keys:
        ref_text = normalize_for_metric(references[key], remove_punct=remove_punct)
        hyp_text = normalize_for_metric(predictions[key], remove_punct=remove_punct)
        ref_words, hyp_words = ref_text.split(), hyp_text.split()
        s, d, ins = levenshtein(ref_words, hyp_words)
        total_s += s; total_d += d; total_i += ins; total_ref_words += len(ref_words)
        cs_, cd_, ci_ = levenshtein(list(ref_text.replace(" ", "")), list(hyp_text.replace(" ", "")))
        char_s += cs_; char_d += cd_; char_i += ci_; total_ref_chars += len(ref_text.replace(" ", ""))
        ref_en, hyp_en = english_tokens(ref_words), english_tokens(hyp_words)
        es, ed, ei = levenshtein(ref_en, hyp_en)
        en_s += es; en_d += ed; en_i += ei; en_ref_words += len(ref_en)
        if viet_set is not None:
            ref_cs = [w for w in ref_words if is_non_vietnamese_word(w, viet_set)]
            hyp_cs = [w for w in hyp_words if is_non_vietnamese_word(w, viet_set)]
            a, b, c = levenshtein(ref_cs, hyp_cs); cs_s += a; cs_d += b; cs_i += c; cs_ref_words += len(ref_cs)
            ref_n = [w for w in ref_words if not is_non_vietnamese_word(w, viet_set)]
            hyp_n = [w for w in hyp_words if not is_non_vietnamese_word(w, viet_set)]
            a, b, c = levenshtein(ref_n, hyp_n); n_s += a; n_d += b; n_i += c; n_ref_words += len(ref_n)
        ref_neg = Counter(w for w in ref_words if w in NEGATION_WORDS)
        hyp_neg = Counter(w for w in hyp_words if w in NEGATION_WORDS)
        missed = sum(max(0, ref_neg[w] - hyp_neg[w]) for w in ref_neg)
        neg_missed += missed; neg_ref_count += sum(ref_neg.values()); neg_utterances += 1 if ref_neg else 0
        per_sample.append({"sample_id": key, "ref_words": len(ref_words), "word_errors": s+d+ins, "substitutions": s, "deletions": d, "insertions": ins, "english_ref_words": len(ref_en), "negation_missed": missed, "reference": references[key], "prediction": predictions[key]})
    def rate(num, den): return None if den == 0 else num / den
    metrics = {
        "matched": len(common_keys), "total_ref": len(references), "total_hyp": len(predictions),
        "missing_in_prediction_count": len(missing_in_pred), "missing_in_reference_count": len(missing_in_ref),
        "missing_in_prediction_preview": missing_in_pred[:20], "missing_in_reference_preview": missing_in_ref[:20],
        "wer": rate(total_s+total_d+total_i, total_ref_words), "cer": rate(char_s+char_d+char_i, total_ref_chars),
        "substitutions": total_s, "deletions": total_d, "insertions": total_i, "ref_words": total_ref_words, "ref_chars": total_ref_chars,
        "english_token_wer": rate(en_s+en_d+en_i, en_ref_words), "english_token_ref_words": en_ref_words,
        "cs_wer": rate(cs_s+cs_d+cs_i, cs_ref_words) if viet_set is not None else None, "cs_ref_words": cs_ref_words,
        "n_wer": rate(n_s+n_d+n_i, n_ref_words) if viet_set is not None else None, "n_ref_words": n_ref_words,
        "negation_miss_rate": rate(neg_missed, neg_ref_count), "negation_missed": neg_missed, "negation_ref_count": neg_ref_count, "negation_utterances": neg_utterances,
        "remove_punctuation_for_metric": remove_punct,
    }
    return {"metrics": metrics, "per_sample": per_sample}

print("[OK] Benchmark functions are ready.")


In [ ]:
# ============================================================
# 7. INFERENCE FUNCTIONS
# ============================================================
import time
import json
from pathlib import Path
from datetime import datetime, timezone
from typing import Any
import torch
from transformers import pipeline


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def read_manifest(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8", errors="replace") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if line:
                row = json.loads(line); row["_line_no"] = line_no; rows.append(row)
    return rows


def load_done_ids(pred_path: Path) -> set[str]:
    done = set()
    if not pred_path.exists(): return done
    with pred_path.open("r", encoding="utf-8", errors="replace") as f:
        for line in f:
            try: row = json.loads(line)
            except Exception: continue
            sid = row.get("sample_id")
            if sid: done.add(str(sid))
    return done


def export_prediction_tsv(pred_jsonl: Path, pred_tsv: Path) -> int:
    n = 0
    pred_tsv.parent.mkdir(parents=True, exist_ok=True)
    with pred_jsonl.open("r", encoding="utf-8", errors="replace") as f, pred_tsv.open("w", encoding="utf-8") as out:
        for line in f:
            if not line.strip(): continue
            row = json.loads(line)
            sid = clean_tsv_cell(row.get("sample_id") or row.get("audio_basename") or f"sample_{n:06d}")
            hyp = clean_tsv_cell(row.get("prediction_text", ""))
            out.write(f"{sid}\t{hyp}\n"); n += 1
    return n


def build_asr_pipeline(model_path: str):
    use_cuda = torch.cuda.is_available()
    device = 0 if use_cuda else -1
    kwargs = {"model": model_path, "tokenizer": model_path, "feature_extractor": model_path, "device": device}
    if use_cuda: kwargs["torch_dtype"] = torch.float16
    print(f"[INFO] Loading ASR model: {model_path}")
    print(f"[INFO] device={device} cuda={use_cuda}")
    return pipeline("automatic-speech-recognition", **kwargs)


def run_asr_inference(manifest: Path, model_path: str, output_predictions: Path, output_prediction_tsv: Path, output_metrics: Path, max_samples: int | None = None, resume: bool = True, log_every: int = 5, language: str = "vi", task: str = "transcribe", model_label: str | None = None):
    rows = read_manifest(manifest)
    if max_samples is not None: rows = rows[:max_samples]
    if not rows: raise RuntimeError("Manifest rỗng, không có gì để inference.")
    output_predictions.parent.mkdir(parents=True, exist_ok=True); output_metrics.parent.mkdir(parents=True, exist_ok=True)
    done_ids = load_done_ids(output_predictions) if resume else set()
    mode = "a" if resume and output_predictions.exists() else "w"
    print("=== INFERENCE CONFIG ===")
    print("manifest:", manifest); print("rows:", len(rows)); print("already_done:", len(done_ids)); print("resume:", resume); print("prediction jsonl:", output_predictions)
    asr = build_asr_pipeline(model_path)
    started = time.time(); n_new = 0; n_error = 0; runtimes = []
    with output_predictions.open(mode, encoding="utf-8") as out:
        for idx, row in enumerate(rows, start=1):
            sid = str(row.get("sample_id") or row.get("audio_basename") or f"sample_{idx:06d}")
            if sid in done_ids: continue
            audio = row.get("audio"); ref = row.get("reference_text", "")
            t0 = time.time(); err = None; hyp = ""
            try:
                result = asr(str(audio), generate_kwargs={"language": language, "task": task})
                hyp = result.get("text", "") if isinstance(result, dict) else str(result)
            except Exception as exc:
                err = repr(exc); n_error += 1
            runtime = time.time() - t0; runtimes.append(runtime); n_new += 1
            pred_row = {"sample_id": sid, "audio": audio, "reference_text": ref, "prediction_text": hyp, "model": model_path, "model_label": model_label or model_path, "runtime_seconds": round(runtime, 4), "error": err, "original_ref_key": row.get("original_ref_key"), "created_at": now_iso()}
            out.write(json.dumps(pred_row, ensure_ascii=False) + "\n"); out.flush()
            if n_new == 1 or n_new % log_every == 0 or idx == len(rows):
                elapsed = time.time() - started; avg = elapsed / max(n_new, 1); remaining = max(len(rows) - idx, 0); eta = remaining * avg
                print("-" * 80, flush=True)
                print(f"[PROGRESS] new={n_new} row={idx}/{len(rows)} elapsed={elapsed:.1f}s avg={avg:.2f}s eta≈{eta/60:.1f}min", flush=True)
                print(f"[SAMPLE] {sid} runtime={runtime:.2f}s error={err}", flush=True)
                print("[REF]", str(ref)[:180], flush=True)
                print("[HYP]", str(hyp)[:180], flush=True)
    n_tsv = export_prediction_tsv(output_predictions, output_prediction_tsv)
    total_runtime = time.time() - started
    metrics = {"created_at": now_iso(), "manifest": str(manifest), "model": model_path, "model_label": model_label or model_path, "n_manifest_rows": len(rows), "n_newly_processed": n_new, "n_prediction_tsv_rows": n_tsv, "n_errors_new": n_error, "resume": resume, "cuda_available": torch.cuda.is_available(), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "total_runtime_seconds": round(total_runtime, 4), "mean_runtime_seconds_per_new_sample": round(sum(runtimes) / len(runtimes), 4) if runtimes else None, "predictions_jsonl": str(output_predictions), "prediction_tsv": str(output_prediction_tsv)}
    output_metrics.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
    print("\n=== INFERENCE DONE ===")
    print(json.dumps(metrics, ensure_ascii=False, indent=2))
    return metrics

print("[OK] Inference functions are ready.")


In [ ]:
# ============================================================
# 8. SMOKE TEST 3 MẪU
# ============================================================
smoke_dir = PRED_DIR / "smoke_3"
smoke_dir.mkdir(parents=True, exist_ok=True)
smoke_metrics = run_asr_inference(
    manifest=manifest_path,
    model_path=MODEL_PATH,
    output_predictions=smoke_dir / "predictions_smoke_3.jsonl",
    output_prediction_tsv=smoke_dir / "prediction_smoke_3.tsv",
    output_metrics=smoke_dir / "inference_metrics_smoke_3.json",
    max_samples=SMOKE_MAX_SAMPLES,
    resume=False,
    log_every=1,
    language=ASR_LANGUAGE,
    task=ASR_TASK,
    model_label=RUN_LABEL,
)


In [ ]:
# ============================================================
# 9. FULL INFERENCE TRÊN TOÀN BỘ TESTSET
# ============================================================
# Nếu notebook bị dừng giữa chừng, giữ RESUME=True rồi chạy lại cell này.
full_pred_jsonl = PRED_DIR / "predictions.jsonl"
full_pred_tsv = PRED_DIR / "prediction.tsv"
full_infer_metrics = PRED_DIR / "inference_metrics.json"

full_metrics = run_asr_inference(
    manifest=manifest_path,
    model_path=MODEL_PATH,
    output_predictions=full_pred_jsonl,
    output_prediction_tsv=full_pred_tsv,
    output_metrics=full_infer_metrics,
    max_samples=MAX_SAMPLES_FOR_FULL_RUN,
    resume=RESUME,
    log_every=LOG_EVERY,
    language=ASR_LANGUAGE,
    task=ASR_TASK,
    model_label=RUN_LABEL,
)


In [ ]:
# ============================================================
# 10. CHẠY BENCHMARK WER/CER/CS-WER/NEGATION MISS RATE
# ============================================================
reference_tsv = reference_tsv_path
prediction_tsv = full_pred_tsv
references = load_tsv(reference_tsv)
predictions = load_tsv(prediction_tsv)
viet_set = load_viet_syllables(VIET_SYLLABLES_PATH)

strict_result = compute_metrics(references, predictions, viet_set=viet_set, remove_punct=False)
normalized_result = compute_metrics(references, predictions, viet_set=viet_set, remove_punct=True)
benchmark_output = {"created_at": now_iso(), "run_label": RUN_LABEL, "model": MODEL_PATH, "reference_tsv": str(reference_tsv), "prediction_tsv": str(prediction_tsv), "viet_syllables_path": str(VIET_SYLLABLES_PATH) if VIET_SYLLABLES_PATH else None, "strict": strict_result["metrics"], "normalized": normalized_result["metrics"]}
benchmark_metrics_path = REPORT_DIR / "benchmark_metrics.json"
per_sample_path = REPORT_DIR / "benchmark_per_sample_errors.jsonl"
benchmark_metrics_path.write_text(json.dumps(benchmark_output, ensure_ascii=False, indent=2), encoding="utf-8")
with per_sample_path.open("w", encoding="utf-8") as f:
    for row in strict_result["per_sample"]: f.write(json.dumps(row, ensure_ascii=False) + "\n")

def fmt_pct(x): return "N/A" if x is None else f"{x * 100:.2f}%"

print("=" * 80)
print("ASR BENCHMARK RESULTS — STRICT")
print("=" * 80)
for k in ["matched", "total_ref", "total_hyp", "missing_in_prediction_count", "missing_in_reference_count"]:
    print(f"{k:35s}: {strict_result['metrics'][k]}")
print("-" * 80)
print(f"WER{'':32s}: {fmt_pct(strict_result['metrics']['wer'])}")
print(f"CER{'':32s}: {fmt_pct(strict_result['metrics']['cer'])}")
print(f"English-token WER{'':20s}: {fmt_pct(strict_result['metrics']['english_token_wer'])}")
print(f"CS-WER via syllable list{'':13s}: {fmt_pct(strict_result['metrics']['cs_wer'])}")
print(f"N-WER via syllable list{'':14s}: {fmt_pct(strict_result['metrics']['n_wer'])}")
print(f"Negation miss rate{'':19s}: {fmt_pct(strict_result['metrics']['negation_miss_rate'])}")
print("-" * 80)
print("Word errors:", strict_result['metrics']['substitutions'] + strict_result['metrics']['deletions'] + strict_result['metrics']['insertions'])
print("Substitutions:", strict_result['metrics']['substitutions'])
print("Deletions:", strict_result['metrics']['deletions'])
print("Insertions:", strict_result['metrics']['insertions'])
print("Reference words:", strict_result['metrics']['ref_words'])
print("=" * 80)
print("\nSaved:")
print("-", benchmark_metrics_path)
print("-", per_sample_path)


In [ ]:
# ============================================================
# 11. XUẤT REPORT MARKDOWN CHO SẾP ĐỌC
# ============================================================
report_path = REPORT_DIR / "BENCHMARK_REPORT.md"
m = strict_result["metrics"]
mn = normalized_result["metrics"]
inf = json.loads(full_infer_metrics.read_text(encoding="utf-8")) if full_infer_metrics.exists() else {}
prep = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else {}

report = f"""# Mentor Test Benchmark Report

## 1. Mục tiêu

Đánh giá model ASR được chỉ định trên bộ test do mentor/sếp cung cấp.  
Notebook này chỉ thực hiện benchmark, không fine-tune và không trộn dữ liệu test vào train.

## 2. Input

| Hạng mục | Giá trị |
|---|---|
| Testset root | `{TESTSET_ROOT}` |
| Transcript | `{TRANSCRIPT_PATH}` |
| Audio dir | `{AUDIO_DIR}` |
| Model | `{MODEL_PATH}` |
| Run label | `{RUN_LABEL}` |

## 3. Chuẩn hóa dữ liệu

| Hạng mục | Số lượng |
|---|---:|
| Audio `.wav` tìm thấy | {prep.get('n_audio_files')} |
| Dòng transcript đọc được | {prep.get('n_transcript_rows')} |
| Audio/transcript khớp được | {prep.get('n_matched')} |
| Transcript không khớp audio | {prep.get('n_unmatched_refs')} |
| Audio không có transcript | {prep.get('n_unreferenced_audio')} |

## 4. Inference

| Hạng mục | Giá trị |
|---|---:|
| Số dòng manifest | {inf.get('n_manifest_rows')} |
| Số mẫu mới đã xử lý trong lần chạy này | {inf.get('n_newly_processed')} |
| Số dòng prediction TSV | {inf.get('n_prediction_tsv_rows')} |
| Số lỗi inference mới | {inf.get('n_errors_new')} |
| CUDA available | {inf.get('cuda_available')} |
| GPU | {inf.get('gpu')} |
| Tổng thời gian inference | {inf.get('total_runtime_seconds')} giây |
| Trung bình / mẫu mới | {inf.get('mean_runtime_seconds_per_new_sample')} giây |

## 5. Benchmark chính — strict metric

| Metric | Giá trị |
|---|---:|
| Matched files | {m.get('matched')} |
| Reference total | {m.get('total_ref')} |
| Prediction total | {m.get('total_hyp')} |
| Missing in prediction | {m.get('missing_in_prediction_count')} |
| Missing in reference | {m.get('missing_in_reference_count')} |
| WER | {fmt_pct(m.get('wer'))} |
| CER | {fmt_pct(m.get('cer'))} |
| English-token WER | {fmt_pct(m.get('english_token_wer'))} |
| CS-WER qua syllable list | {fmt_pct(m.get('cs_wer'))} |
| N-WER qua syllable list | {fmt_pct(m.get('n_wer'))} |
| Negation miss rate | {fmt_pct(m.get('negation_miss_rate'))} |

## 6. Benchmark phụ — normalized metric

Metric này xóa punctuation trước khi tính, chỉ dùng để phân tích phụ.

| Metric | Giá trị |
|---|---:|
| WER normalized | {fmt_pct(mn.get('wer'))} |
| CER normalized | {fmt_pct(mn.get('cer'))} |
| English-token WER normalized | {fmt_pct(mn.get('english_token_wer'))} |
| Negation miss rate normalized | {fmt_pct(mn.get('negation_miss_rate'))} |

## 7. Output files

| File | Path |
|---|---|
| Manifest JSONL | `{manifest_path}` |
| Reference TSV | `{reference_tsv_path}` |
| Prediction JSONL | `{full_pred_jsonl}` |
| Prediction TSV | `{full_pred_tsv}` |
| Inference metrics | `{full_infer_metrics}` |
| Benchmark metrics | `{benchmark_metrics_path}` |
| Per-sample error JSONL | `{per_sample_path}` |

## 8. Ghi chú diễn giải

- `WER/CER` là metric tổng thể của ASR.
- `English-token WER` là metric phụ để quan sát lỗi trên các token có chữ Latin như tên thuốc, xét nghiệm, thuật ngữ tiếng Anh.
- `CS-WER` chỉ có giá trị nếu cung cấp file danh sách âm tiết tiếng Việt qua biến `VIET_SYLLABLES_PATH`.
- `Negation miss rate` đo tỉ lệ model bỏ sót các từ phủ định như `không`, `chưa`, `chẳng`, `chả`, `đừng`, `chớ`.
- Kết quả này là benchmark trên test set mentor cung cấp, không dùng để train hoặc chọn hyperparameter.
"""
report_path.write_text(report, encoding="utf-8")
print(report)
print("\n[DONE] Report saved to:", report_path)


In [ ]:
# ============================================================
# 12. ĐÓNG GÓI ARTIFACTS ĐỂ GỬI LẠI
# ============================================================
# Bundle này không chứa audio gốc và không chứa model. Nó chỉ chứa manifest/reference/prediction/metrics/report.
import tarfile
bundle_path = OUTPUT_ROOT / f"{RUN_LABEL}_benchmark_artifacts.tar.gz"
files_to_include = [manifest_path, reference_tsv_path, summary_path, prepare_report_path, full_pred_jsonl, full_pred_tsv, full_infer_metrics, benchmark_metrics_path, per_sample_path, report_path]
with tarfile.open(bundle_path, "w:gz") as tar:
    for p in files_to_include:
        if p.exists(): tar.add(p, arcname=str(p.relative_to(RUN_DIR)))
print("[DONE] Bundle created:")
print(bundle_path)
print("size MB:", round(bundle_path.stat().st_size / (1024 * 1024), 3))


## 13. Kết luận sử dụng notebook

Sau khi chạy xong, anh xem nhanh các file sau:

```text
BENCHMARK_REPORT.md
benchmark_metrics.json
prediction.tsv
predictions.jsonl
```

Nếu cần chạy lại từ đầu:

1. Xóa thư mục output của run hiện tại, hoặc đổi `RUN_LABEL`.
2. Giữ `RESUME=True` nếu chỉ muốn chạy tiếp phần bị dừng.
3. Nếu muốn smoke test lại, chạy riêng cell smoke test.

Notebook này chỉ phục vụ benchmark external test set do mentor/sếp cung cấp, không chứa logic fine-tune ViMedCSS.
